# DEASUS · Documentos de Acompanhamento Periódico do Auditor

**Como usar (3 passos):**

1. No BI, aba **Acompanhamento Periódico**: filtre o período → na tabela, menu **…** → **Exportar dados** → **Dados com layout atual** → `.xlsx`.
2. Preencha o **período** e os **avaliadores** abaixo.
3. Menu **Ambiente de execução → Executar tudo**. Quando aparecer o botão **Escolher arquivos**, selecione **juntos** o Excel exportado e o modelo `.docx`. No fim, o navegador baixa um `.zip` com um documento por auditor.

Dica: se quiser a avaliação descritiva já preenchida, acrescente no Excel uma coluna chamada **Avaliação Descritiva** antes de enviar. Se não, escreva direto em cada documento depois.

In [ ]:
#@title 1. Dados do período { display-mode: "form" }
PERIODO = "1º/2026"  #@param {type:"string"}
AVALIADORES = ""  #@param {type:"string"}


In [ ]:
#@title 2. Preparação (não precisa mexer) { display-mode: "form" }
!pip -q install docxtpl
# código do gerador (fonte: gerar_documentos.py no repositório BI-Interno-DEASUS)
CODIGO = '"""Gera os Documentos de Acompanhamento Periódico do Auditor (um .docx por auditor).\n\nEntrada:\n  1. O Excel exportado da tabela da aba "Acompanhamento Periódico" do BI\n     (menu ... da tabela -> Exportar dados -> Dados com layout atual -> .xlsx).\n  2. O modelo .docx com os campos {{ Auditor }}, {{ Quadrimestre }}, {{ Avaliadores }},\n     {{ atividade_1 }} ... {{ avaliacao_5 }}, {{ proas }} e {{ Avaliacao_Descritiva }}.\n\nAs colunas são lidas PELO NOME (não pela posição). Se alguém renomear uma coluna no BI,\najuste a lista ATIVIDADES abaixo.\n\nUso no computador (opcional; no dia a dia use o notebook no Colab):\n    python gerar_documentos.py exportacao.xlsx modelo.docx --periodo "1º/2026" --avaliadores "Fulano, Beltrano"\n"""\nimport os\nimport re\nimport ast\nimport zipfile\nimport unicodedata\nfrom datetime import datetime\n\nimport pandas as pd\nfrom docxtpl import DocxTemplate\n\n# Linhas da tabela do documento: (texto que aparece no documento, coluna do Excel exportado).\n# O modelo tem 5 linhas (atividade_1..5). Para mudar o que aparece, troque aqui.\nATIVIDADES = [\n    ("Total de auditorias designadas", "Total de auditorias designadas"),\n    ("Auditorias (coordenação)", "Auditorias (coordenação)"),\n    ("Auditorias (composição de equipe)", "Auditorias (composição de equipe)"),\n    ("Relatórios Preliminares entregues", "Relatórios Preliminares entregues"),\n    ("Qtd de Matrizes RAI", "Qtd de Matrizes RAI"),   # 5ª linha igual à do notebook antigo\n]\nCOL_AUDITOR = "Auditor"\nCOL_PROAS = "PROAs"\nCOL_DESCRITIVA = "Avaliação Descritiva"   # opcional: se existir no Excel, vai para o documento\n\nMESES = ["janeiro", "fevereiro", "março", "abril", "maio", "junho", "julho",\n         "agosto", "setembro", "outubro", "novembro", "dezembro"]\n\n\ndef _chave(txt):\n    """Compara nomes de coluna sem acento, maiúsculas ou espaços extras."""\n    txt = unicodedata.normalize("NFD", str(txt))\n    txt = "".join(c for c in txt if unicodedata.category(c) != "Mn")\n    return re.sub(r"\\s+", " ", txt).strip().lower()\n\n\ndef ler_exportacao(caminho):\n    """Lê o Excel/CSV exportado do BI. Acha a linha de cabeçalho (a que tem \'Auditor\')\n    e descarta linhas vazias, de total e as notas de filtro que o Power BI põe no fim."""\n    if caminho.lower().endswith(".csv"):\n        bruto = pd.read_csv(caminho, header=None, dtype=object)\n    else:\n        bruto = pd.read_excel(caminho, header=None, dtype=object)\n    linha_cab = None\n    for i, linha in bruto.iterrows():\n        if any(_chave(v) == _chave(COL_AUDITOR) for v in linha.values if pd.notna(v)):\n            linha_cab = i\n            break\n    if linha_cab is None:\n        raise ValueError(f"Não achei a coluna \'{COL_AUDITOR}\' no arquivo {caminho}. "\n                         "Confira se exportou a tabela da aba Acompanhamento Periódico.")\n    df = bruto.iloc[linha_cab + 1:].copy()\n    df.columns = [str(c).strip() if pd.notna(c) else f"_col{j}" for j, c in enumerate(bruto.iloc[linha_cab])]\n    mapa = {_chave(c): c for c in df.columns}\n    df = df.rename(columns={mapa[_chave(COL_AUDITOR)]: COL_AUDITOR})\n    df[COL_AUDITOR] = df[COL_AUDITOR].astype("string").str.strip()\n    ruido = df[COL_AUDITOR].isna() | (df[COL_AUDITOR] == "") | df[COL_AUDITOR].str.lower().isin(["total"]) \\\n        | df[COL_AUDITOR].str.lower().str.startswith(("filtros aplicados", "applied filters"))\n    return df[~ruido].reset_index(drop=True), mapa\n\n\ndef _numero(v):\n    if v is None or (isinstance(v, float) and pd.isna(v)) or (isinstance(v, str) and not v.strip()):\n        return "0"\n    try:\n        f = float(str(v).replace(",", "."))\n        return str(int(f)) if f.is_integer() else f"{f:.2f}".replace(".", ",")\n    except ValueError:\n        return str(v)\n\n\ndef _proas(v):\n    """Aceita \'A; B; C\' (BI) ou "[\'A\', \'B\']" (planilha antiga do notebook)."""\n    if v is None or (isinstance(v, float) and pd.isna(v)):\n        return ""\n    s = str(v).strip()\n    if s.startswith("["):\n        try:\n            itens = ast.literal_eval(s)\n        except (ValueError, SyntaxError):\n            itens = [s]\n    else:\n        itens = re.split(r";|\\n", s)\n    return "\\n".join(re.sub(r"\\s+", " ", i).strip() for i in itens if str(i).strip())\n\n\ndef montar_contexto(linha, mapa, periodo, avaliadores):\n    hoje = datetime.now()\n    ctx = {\n        "Auditor": linha[COL_AUDITOR],\n        "Quadrimestre": periodo,\n        "Avaliadores": avaliadores,\n        "DataHoje": f"{hoje.day} de {MESES[hoje.month - 1]} de {hoje.year}",\n        "proas": _proas(linha.get(mapa.get(_chave(COL_PROAS)))),\n        "Avaliacao_Descritiva": "",\n    }\n    col_desc = mapa.get(_chave(COL_DESCRITIVA))\n    if col_desc and pd.notna(linha.get(col_desc)):\n        ctx["Avaliacao_Descritiva"] = str(linha.get(col_desc))\n    for i, (rotulo, coluna) in enumerate(ATIVIDADES, start=1):\n        col = mapa.get(_chave(coluna))\n        ctx[f"atividade_{i}"] = rotulo\n        ctx[f"avaliacao_{i}"] = _numero(linha.get(col)) if col else ""\n    return ctx\n\n\ndef gerar(caminho_excel, caminho_modelo, periodo, avaliadores="", pasta_saida="documentos_acompanhamento"):\n    df, mapa = ler_exportacao(caminho_excel)\n    faltando = [c for _, c in ATIVIDADES if _chave(c) not in mapa]\n    if faltando:\n        print("ATENÇÃO: colunas não encontradas no Excel (ficarão em branco no documento):", faltando)\n    if _chave(COL_PROAS) not in mapa:\n        print(f"ATENÇÃO: coluna \'{COL_PROAS}\' não encontrada; a lista de PROAs sairá vazia.")\n    os.makedirs(pasta_saida, exist_ok=True)\n    arquivos = []\n    for n, linha in df.iterrows():\n        ctx = montar_contexto(linha, mapa, periodo, avaliadores)\n        doc = DocxTemplate(caminho_modelo)\n        doc.render(ctx)\n        nome = re.sub(r"[^\\w\\-]+", "_", ctx["Auditor"]).strip("_")\n        caminho = os.path.join(pasta_saida, f"acompanhamento_{n + 1:02d}_{nome}.docx")\n        doc.save(caminho)\n        arquivos.append(caminho)\n        print("Gerado:", caminho)\n    zip_path = pasta_saida.rstrip("/\\\\") + ".zip"\n    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:\n        for a in arquivos:\n            z.write(a, os.path.basename(a))\n    print(f"\\n{len(arquivos)} documento(s) gerado(s). ZIP: {zip_path}")\n    return zip_path\n\n\nif __name__ == "__main__":\n    import argparse\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("excel")\n    ap.add_argument("modelo")\n    ap.add_argument("--periodo", required=True, help=\'Ex.: "1º/2026"\')\n    ap.add_argument("--avaliadores", default="")\n    ap.add_argument("--saida", default="documentos_acompanhamento")\n    a = ap.parse_args()\n    gerar(a.excel, a.modelo, a.periodo, a.avaliadores, a.saida)\n'
open("gerar_documentos.py", "w", encoding="utf-8").write(CODIGO)
print("Pronto.")


In [ ]:
#@title 3. Enviar arquivos e gerar documentos { display-mode: "form" }
import os, shutil
from google.colab import files
import gerar_documentos as g

for pasta in ("entrada", "documentos_acompanhamento"):
    shutil.rmtree(pasta, ignore_errors=True)
os.makedirs("entrada")
print("Selecione JUNTOS o Excel exportado do BI e o modelo .docx:")
enviados = files.upload()
excel = modelo = None
for nome in enviados:
    destino = os.path.join("entrada", nome)
    shutil.move(nome, destino)
    if nome.lower().endswith((".xlsx", ".csv")):
        excel = destino
    elif nome.lower().endswith(".docx"):
        modelo = destino
if not excel or not modelo:
    raise SystemExit("Faltou arquivo: envie o Excel (.xlsx) E o modelo (.docx). Rode esta célula de novo.")
zip_path = g.gerar(excel, modelo, PERIODO, AVALIADORES)
files.download(zip_path)
